# Choosing the Optimal Data Management Architecture

## Background

In this exercise, you act as a data engineer tasked with recommending the most cost-efficient data management architecture for a company.

**Group competition (in class).** Each group calculates the 2-year costs for **one** architecture, the one your group explained in the expert round. The comparison table is assembled on the whiteboard from all groups. Afterwards every group gives a recommendation, with one rule: **you may not recommend your own architecture.**

At home: calculate all four and check the whiteboard numbers (bonus below).

**Background.** The company generates and uses data in various ways, and the decision must be made considering only the costs of storage, data transfer, querying, maintenance/administration, and data source acquisition, along with the frequency of these operations. In this exercise, we focus only on costs.
In real-life scenarios, cost is not the only factor and trade-offs have to be made.

The company is deciding between:
1. Data Warehouse
2.	Data Lake
3.	Hybrid Data Lake
4.	Lakehouse

Each architecture has different cost structures and performance trade-offs.

In [ ]:
# Your group's architecture: "Data Warehouse", "Data Lake", "Hybrid Data Lake" or "Lakehouse"
MY_ARCHITECTURE = "Data Lake"

In the following pandas dataframe the costs for the here considered operations are provided.

In [64]:
import pandas as pd

costs_information_dict = {
    "architecture": ["Data Warehouse", "Data Lake", "Hybrid Data Lake", "Lakehouse"],
    "Storage Cost ($/TB/month)": [300, 50, 350, 80],
    "Transfer Cost ($/GB)": [0.1,0.1,0.1,0.1],
    "Query Cost ($/1,000 queries)": [0.5, 1, 0.5, 0.75],
    "Maintenance Cost ($/month)": [50, 30, 70, 30],
}

costs = pd.DataFrame(costs_information_dict)
costs = costs.set_index('architecture')

costs

,Storage Cost ($/TB/month),Transfer Cost ($/GB),"Query Cost ($/1,000 queries)",Maintenance Cost ($/month)
architecture,,,,
Data Warehouse,300,0.1,0.50,50
Data Lake,50,0.1,1.00,30
Hybrid Data Lake,350,0.1,0.50,70
Lakehouse,80,0.1,0.75,30


## Usage Information

1. **Data sources** (acquisition cost per GB, paid to the source; plus transfer cost per GB, see cost table)
    - Source A, web tracking: produces 10 GB of data **daily**, acquisition cost $0.2/GB
    - Source B, shop backend export: produces 100 GB of data **weekly**, acquisition cost $0.1/GB
    - Source C, **purchased financial market data feed**: 20 GB per **month**, acquisition cost **$20/GB**. The vendor bills per download: **every re-download of historical data costs the full price again.**
2. **Data storage** (storage cost per GB stored, billed every month)
    - Storage starts empty and grows by **20 GB per month**, data is kept for the whole 2 years
    - so month 1 bills 20 GB, month 2 bills 40 GB, ..., month 24 bills 480 GB
3. **Query patterns** (query cost per 1,000 queries)
    - Type 1 queries: run 1,000 times per **day**
    - Type 2 queries: run 10,000 times per **week**
    - Type 3 queries: run 15,000 times per **month**
4. **Maintenance**: fixed cost per month, see cost table

Use 2 years = 730 days, a week = 7 days, a month = 365/12 days.

## Task

### 1. Calculate the 2-year costs of your architecture (10 min)

For `MY_ARCHITECTURE`, calculate the total cost within 2 years of

- data source acquisition
- data storage
- data transfer
- querying the data
- maintenance/administration

and the overall sum. Print one line like `Data Lake: 1234 $ in 2 years (acquisition ..., storage ..., transfer ..., queries ..., maintenance ...)`.

### 2. Whiteboard

Write your overall number and the five components on the whiteboard. Compare with the other groups: same assumptions, same numbers?

### 3. Recommendation (2 min per group)

Based on the whiteboard table: which architecture would you recommend to the company? **Not your own.** Give one cost argument and one non-cost argument (performance, flexibility, risk of a data swamp, team skills, ...).

**Extra question for the recommendation:** after one year the schema of the cleaned tables has to change and all history must be rebuilt from raw data. Which architecture pays the financial data feed a **second time**, and which ones do not? Why?

In [ ]:
costs["Storage Cost ($/GB/month)"] = costs["Storage Cost ($/TB/month)"] / 1000

DAYS = 2 * 365
DAYS_PER_MONTH = 365 / 12

# daily volumes and acquisition prices of the three sources
gb_per_day = {"A": 10, "B": 100 / 7, "C": 20 / DAYS_PER_MONTH}
price_per_gb = {"A": 0.2, "B": 0.1, "C": 20}

acquisition_per_day = sum(gb_per_day[s] * price_per_gb[s] for s in gb_per_day)
transfer_gb_per_day = sum(gb_per_day.values())
storage_gb_months = 20 * sum(range(1, 25))                        # 20 + 40 + ... + 480 = 6000
queries_count = DAYS * (1000 + 10000 / 7 + 15000 / DAYS_PER_MONTH)


def two_year_costs(architecture):
    c = costs.loc[architecture]
    acquisition = acquisition_per_day * DAYS
    storage = c["Storage Cost ($/GB/month)"] * storage_gb_months
    transfer = c["Transfer Cost ($/GB)"] * transfer_gb_per_day * DAYS
    queries = c["Query Cost ($/1,000 queries)"] * queries_count / 1000
    maintenance = c["Maintenance Cost ($/month)"] * 24
    return {"acquisition": acquisition, "storage": storage, "transfer": transfer,
            "queries": queries, "maintenance": maintenance,
            "total": acquisition + storage + transfer + queries + maintenance}


mine = two_year_costs(MY_ARCHITECTURE)
print(f"{MY_ARCHITECTURE}: {mine['total']:.0f} $ in 2 years (acquisition {mine['acquisition']:.0f}, "
      f"storage {mine['storage']:.0f}, transfer {mine['transfer']:.0f}, queries {mine['queries']:.0f}, "
      f"maintenance {mine['maintenance']:.0f})")

## At home: all four architectures

Loop over all architectures, build one table (rows = architectures, columns = the five cost components + total) and sort it by total. Compare with the whiteboard photo.

In [ ]:
# all four, this is what the whiteboard table should look like
comparison = pd.DataFrame({a: two_year_costs(a) for a in costs.index}).T.round(0)
comparison.sort_values("total")

### Recommendation (example)

Within 2 years the **Lakehouse** is the cheapest option, mainly because cheap object storage dominates and the query engine is only moderately more expensive than a warehouse. The plain **Data Lake** is close behind on cost, but every query is more expensive and there is no built-in query execution: the risk of ending up with a data swamp is real if nobody owns the structure. The **Data Warehouse** costs about 8 % more than the lakehouse and buys fast, governed queries, which matters if the main consumers are dashboards with many users. The **Hybrid Data Lake** is the most expensive because data is stored twice and two systems have to be maintained; it is only advisable if low-latency operational BI *and* cheap exploration are both hard requirements.

Note that the biggest single item is the purchased financial data feed, and it is the same for all four architectures, as long as nothing goes wrong.

**Extra question.** The **Data Warehouse** pays the feed a second time: it keeps only the cleaned tables, so rebuilding the history after a schema change means downloading 12 months of source C again at full price (see the bonus below for the number). **Lake, Hybrid and Lakehouse** keep the raw data, they only re-run the transformation. Expensive purchased data is the strongest argument for keeping raw data somewhere, even if you otherwise run a warehouse.

Beyond cost: team skills (who can run a query engine on object storage?), the share of unstructured data (images, documents for AI), and how often schemas change decide the case in practice.

# Bonus Exercise (at home)

Two things go wrong in the two years.

1. **Incident after 6 months.** An error in the transformation process destroys all cleaned data (staging and marts). There was no backup. You rebuild the cleaned data and, lesson learned, from month 7 on you keep a full copy of your storage (assume the storage price doubles from month 7 to month 24).
2. **Schema update after 12 months.** The cleaned tables need a new schema and the whole history has to be rebuilt from raw data.

Rebuilding from raw data is free in our model if the raw data is still in your own storage (the transformation runs again). If it is **not** (a pure warehouse keeps only cleaned tables), the raw data has to be re-loaded from the sources: transfer costs again, and acquisition costs again (source C bills every download at full price; assume A and B do as well). The sources still have the history.

## Task

Calculate the 2-year costs for all four architectures including both events. Which architecture became much more expensive, and why is that not visible in the cost table at the top?

In [ ]:
MONTHS_INCIDENT, MONTHS_SCHEMA = 6, 12

# extra storage: from month 7 to 24 the stored volume is billed twice
extra_storage_gb_months = 20 * sum(range(7, 25))

# re-loading n months of raw data from the sources (only architectures without raw data)
def reload_costs(months, architecture):
    days = months * DAYS_PER_MONTH
    acquisition = acquisition_per_day * days
    transfer = costs.loc[architecture, "Transfer Cost ($/GB)"] * transfer_gb_per_day * days
    return acquisition, transfer

keeps_raw_data = {"Data Warehouse": False, "Data Lake": True, "Hybrid Data Lake": True, "Lakehouse": True}

rows = {}
for a in costs.index:
    base = two_year_costs(a)
    extra_storage = costs.loc[a, "Storage Cost ($/GB/month)"] * extra_storage_gb_months
    reload_acq, reload_tr = (0, 0)
    if not keeps_raw_data[a]:
        a1, t1 = reload_costs(MONTHS_INCIDENT, a)
        a2, t2 = reload_costs(MONTHS_SCHEMA, a)
        reload_acq, reload_tr = a1 + a2, t1 + t2
    rows[a] = {**base, "extra storage (backup)": extra_storage, "re-load acquisition": reload_acq,
               "re-load transfer": reload_tr,
               "total with events": base["total"] + extra_storage + reload_acq + reload_tr}

with_events = pd.DataFrame(rows).T.round(0)
with_events[["total", "extra storage (backup)", "re-load acquisition", "re-load transfer", "total with events"]].sort_values("total with events")

**Answer.** The Data Warehouse jumps from third to by far the most expensive option: re-loading 6 + 12 months of raw data means paying the financial data feed again, and that single item is larger than the whole difference between the four architectures in the base case. The cost table at the top cannot show this because it prices storage, queries and maintenance per unit; it does not price *what happens when you need the raw data again*. Keeping raw data (lake, lakehouse, hybrid) is an insurance whose premium is cheap object storage.